# Week 7, Day 3: Principal component analysis: directions of greatest variance

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w007/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Write PCA

Write `pca(X, k)` using the SVD of the centred data. Return the top `k` principal components as the rows of a `k` by `p` array, the variance along each, and the mean vector. Test it on 2D data with a known long axis.

In [ ]:
import numpy as np
def pca(X, k):
    mu = X.mean(axis=0)
    U, s, Vt = np.linalg.svd(X - mu, full_matrices=False)
    comps = ...
    var = ...
    return comps, var, mu

rng = np.random.default_rng(0)
Z = rng.normal(size=(500, 2)) * [3.0, 0.5]                       # long axis along x
R = np.array([[np.cos(0.5), -np.sin(0.5)], [np.sin(0.5), np.cos(0.5)]])
X2 = Z @ R.T + [10, -4]                                          # rotate by 0.5 rad and shift
comps, var, mu = pca(X2, 2)
print("PC1:", comps[0].round(3), " expected about", R[:, 0].round(3), "(or its negative)")
print("variances:", var.round(2), " expected about", [9, 0.25])

In [ ]:
import numpy as np
def make(seed):
    r = np.random.default_rng(seed)
    return r.normal(size=(200, 4)) @ r.normal(size=(4, 4)) + r.normal(size=4) * 5

def test_shapes():
    "pca returns k components (rows), k variances and the mean"
    comps, var, mu = pca(make(1), 2)
    assert comps.shape == (2, 4) and var.shape == (2,) and mu.shape == (4,)

def test_against_eigh():
    "components and variances match the eigen-decomposition of the covariance (up to sign)"
    X = make(2)
    comps, var, mu = pca(X, 3)
    lam, V = np.linalg.eigh(np.cov(X, rowvar=False))
    lam, V = lam[::-1], V[:, ::-1]
    assert np.allclose(var, lam[:3])
    assert np.allclose(np.abs(np.sum(comps * V[:, :3].T, axis=1)), 1)

def test_mean():
    "mu is the column mean"
    X = make(3)
    assert np.allclose(pca(X, 1)[2], X.mean(axis=0))

test_shapes()
test_against_eigh()
test_mean()
print("All checks passed")

### Exercise: Project and reconstruct

Write `project(X, comps, mu)`, which returns the scores (one row per observation, one column per component), and `reconstruct(Z, comps, mu)`, which maps scores back into the original space. Check that reconstructing from one component of `X2` leaves a mean squared error equal to the variance of the dropped component (up to the T − 1 versus T factor).

In [ ]:
def project(X, comps, mu):
    return ...

def reconstruct(Z, comps, mu):
    return ...

comps, var, mu = pca(X2, 2)
c1 = comps[:1]
X1 = reconstruct(project(X2, c1, mu), c1, mu)
print("mean squared error with 1 component:", np.mean(np.sum((X2 - X1) ** 2, axis=1)).round(4), "  variance of PC2:", var[1].round(4))

In [ ]:
import numpy as np
def test_project():
    "project gives the scores (X - mu) projected onto the components"
    comps, var, mu = pca(X2, 2)
    assert np.allclose(project(X2, comps, mu), (X2 - mu) @ comps.T)

def test_reconstruct():
    "reconstruct inverts project when all components are kept"
    comps, var, mu = pca(X2, 2)
    assert np.allclose(reconstruct(project(X2, comps, mu), comps, mu), X2)

def test_error():
    "the mean squared reconstruction error with 1 component equals the variance of PC2 times (T-1)/T"
    comps, var, mu = pca(X2, 2)
    c1 = comps[:1]
    err = np.mean(np.sum((X2 - reconstruct(project(X2, c1, mu), c1, mu)) ** 2, axis=1))
    assert np.isclose(err, var[1] * (len(X2) - 1) / len(X2))

test_project()
test_reconstruct()
test_error()
print("All checks passed")

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(73)
t = np.arange(32)
def template(depth, width, recovery):
    return -depth * np.exp(-(t - 10) ** 2 / (2 * width ** 2)) + recovery * np.exp(-(t - 16 - width) ** 2 / (2 * (2 * width) ** 2))
templates = np.array([template(60, 1.5, 15), template(45, 2.5, 25), template(55, 1.2, 5)])
labels = rng.integers(0, 3, 300)                                     # which neuron fired (unknown in practice)
W = templates[labels] * rng.uniform(0.85, 1.15, (300, 1)) + rng.normal(0, 8, (300, 32))   # 300 spikes by 32 samples (µV)
plt.figure(figsize=(5, 3)); plt.plot(t, W[:40].T, color="0.6", lw=0.6); plt.plot(t, templates.T, lw=2)
plt.xlabel("sample (1 sample = 1/30 ms)"); plt.ylabel("voltage (µV)"); plt.show()

### Exercise: Separate the neurons

Compute the first two PC scores of every spike in `scores`. Use the `separation` function provided (mean distance between cluster centres over mean distance of spikes from their own centre) to score the PCs (`sep_pca`) and the trough voltage alone, column 10 (`sep_trough`). The labels are used only to evaluate: in a real recording you would cluster the scores without them (Week 60).

In [ ]:
def separation(F, labels):
    cents = np.array([F[labels == k].mean(axis=0) for k in range(3)])
    within = np.mean([np.linalg.norm(F[labels == k] - cents[k], axis=1).mean() for k in range(3)])
    between = np.mean([np.linalg.norm(cents[i] - cents[j]) for i in range(3) for j in range(i + 1, 3)])
    return between / within

comps, var, mu = ...
scores = ...
sep_pca = ...
sep_trough = ...

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].scatter(scores[:, 0], scores[:, 1], c=labels, s=8, cmap="viridis"); ax[0].set_xlabel("PC1"); ax[0].set_ylabel("PC2")
ax[1].plot(t, comps.T); ax[1].set_title("the components as waveforms"); ax[1].set_xlabel("sample")
plt.tight_layout(); plt.show()
print("separation: PCs", round(sep_pca, 2), " trough alone", round(sep_trough, 2))

In [ ]:
import numpy as np
def test_scores():
    "scores holds the first two PC scores of every spike"
    comps, var, mu = pca(W, 2)
    assert scores.shape == (300, 2)
    assert np.allclose(np.abs(scores), np.abs(project(W, comps, mu)))

def test_sep():
    "separation is the between-centroid distance over within-cluster spread"
    assert np.isclose(sep_pca, separation(scores, labels))
    assert np.isclose(sep_trough, separation(W[:, [10]], labels))

def test_better():
    "the two PCs separate the neurons at least twice as well as the trough voltage"
    assert sep_pca > 2 * sep_trough

test_scores()
test_sep()
test_better()
print("All checks passed")